In [1]:
import os
current_directory = os.getcwd()
print("Your notebook is working from this folder:")
print(current_directory)

Your notebook is working from this folder:
C:\Users\eross\Jupyter Lab Files


In [2]:
# Cell 1: Import Libraries and Setup
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.cluster import DBSCAN
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")
print("Ready to load data...\n")

✓ Libraries imported successfully!
Ready to load data...



In [3]:
# Cell 2: Load and Prepare Data
print("="*80)
print("SJV DAIRY CLUSTER ANALYSIS FOR MOBILE EMISSIONS MEASUREMENT")
print("="*80)

# Load the three CADD CSV files
facility_df = pd.read_csv('CADD v2.0.0 .csv')
herd_df = pd.read_csv('CADD Facility Herd Size V2.csv')
digester_df = pd.read_csv('CADD Anaerobic Digester V2.csv')

print(f"\n✓ Loaded facility data: {len(facility_df):,} dairies")
print(f"✓ Loaded herd size data: {len(herd_df):,} records")
print(f"✓ Loaded digester data: {len(digester_df):,} records")

# Define SJV counties
sjv_counties = ['Fresno', 'Kings', 'Kern', 'Tulare', 'Merced', 'Stanislaus', 'San Joaquin', 'Madera']
print(f"\n✓ SJV counties defined: {', '.join(sjv_counties)}")

SJV DAIRY CLUSTER ANALYSIS FOR MOBILE EMISSIONS MEASUREMENT

✓ Loaded facility data: 2,115 dairies
✓ Loaded herd size data: 23,444 records
✓ Loaded digester data: 174 records

✓ SJV counties defined: Fresno, Kings, Kern, Tulare, Merced, Stanislaus, San Joaquin, Madera


In [4]:
# Cell 3: Define Herd Size Ranges
herd_size_ranges = [
    {'name': 'Small-Mid (1000-2000)', 'min': 1000, 'max': 2000},
    {'name': 'Mid (2000-3500)', 'min': 2000, 'max': 3500},
    {'name': 'Mid-Large (3000-5000)', 'min': 3000, 'max': 5000},
    {'name': 'Large (4000-7000)', 'min': 4000, 'max': 7000},
    {'name': 'Very Large (6000-10000)', 'min': 6000, 'max': 10000},
]

print("\n" + "="*80)
print("HERD SIZE RANGES DEFINED")
print("="*80)
for r in herd_size_ranges:
    print(f"  • {r['name']}: {r['min']:,} - {r['max']:,} milking cows")


HERD SIZE RANGES DEFINED
  • Small-Mid (1000-2000): 1,000 - 2,000 milking cows
  • Mid (2000-3500): 2,000 - 3,500 milking cows
  • Mid-Large (3000-5000): 3,000 - 5,000 milking cows
  • Large (4000-7000): 4,000 - 7,000 milking cows
  • Very Large (6000-10000): 6,000 - 10,000 milking cows


In [5]:
# Cell 4: Filter for SJV & Get Latest Herd Sizes
print("\n" + "="*80)
print("FILTERING FOR SJV COUNTIES & LATEST HERD SIZES")
print("="*80)

# Filter for SJV counties
facility_sjv = facility_df[facility_df['County'].isin(sjv_counties)].copy()
print(f"\n✓ Found {len(facility_sjv):,} dairies in SJV counties")

# Get most recent herd size (2023 preferred, then 2022, etc.)
herd_latest = herd_df.sort_values('Year', ascending=False).drop_duplicates('CADDID', keep='first').copy()
print(f"✓ Got latest herd sizes for {len(herd_latest):,} dairies")

# Merge facility + latest herd data
facility_sjv = facility_sjv.merge(herd_latest, on='CADDID', how='left')
print(f"✓ Merged facility and herd data: {len(facility_sjv):,} records")

# Identify digesters
digester_ids = set(digester_df['CADDID'].unique())
facility_sjv['HasDigester'] = facility_sjv['CADDID'].isin(digester_ids)
print(f"✓ Identified {len(digester_ids)} digesters in SJV")

# Assign herd size categories
def assign_herd_size_range(herd_size, ranges):
    """Assign dairy to a herd size range"""
    if pd.isna(herd_size):
        return None
    for r in ranges:
        if r['min'] <= herd_size <= r['max']:
            return r['name']
    return None

facility_sjv['HerdSizeRange'] = facility_sjv['MilkCows'].apply(
    lambda x: assign_herd_size_range(x, herd_size_ranges)
)

# Filter out dairies with no assigned herd size range
facility_sjv = facility_sjv[facility_sjv['HerdSizeRange'].notna()].copy()
print(f"✓ {len(facility_sjv):,} dairies assigned to herd size ranges")


FILTERING FOR SJV COUNTIES & LATEST HERD SIZES

✓ Found 1,558 dairies in SJV counties
✓ Got latest herd sizes for 2,116 dairies
✓ Merged facility and herd data: 1,558 records
✓ Identified 171 digesters in SJV
✓ 445 dairies assigned to herd size ranges


In [6]:
# Cell 5: Summarize Dairy Distribution
print("\n" + "="*80)
print("DAIRY DISTRIBUTION BY HERD SIZE RANGE & DIGESTER STATUS")
print("="*80)

summary_data = []

for herd_range in herd_size_ranges:
    range_name = herd_range['name']
    range_data = facility_sjv[facility_sjv['HerdSizeRange'] == range_name]
    
    if len(range_data) > 0:
        digester_count = range_data[range_data['HasDigester']].shape[0]
        non_digester_count = range_data[~range_data['HasDigester']].shape[0]
        total = len(range_data)
        
        summary_data.append({
            'Herd Size Range': range_name,
            'With Digesters': digester_count,
            'Without Digesters': non_digester_count,
            'Total': total
        })

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

print("\n" + "-"*80)
total_all = len(facility_sjv)
digesters_all = facility_sjv[facility_sjv['HasDigester']].shape[0]
non_digesters_all = facility_sjv[~facility_sjv['HasDigester']].shape[0]
print(f"\nTOTAL ACROSS ALL RANGES:")
print(f"  With Digesters:     {digesters_all:,}")
print(f"  Without Digesters:  {non_digesters_all:,}")
print(f"  Total:              {total_all:,}")


DAIRY DISTRIBUTION BY HERD SIZE RANGE & DIGESTER STATUS


        Herd Size Range  With Digesters  Without Digesters  Total
  Small-Mid (1000-2000)              36                204    240
        Mid (2000-3500)              74                 72    146
  Mid-Large (3000-5000)              26                 12     38
      Large (4000-7000)              13                  2     15
Very Large (6000-10000)               5                  1      6

--------------------------------------------------------------------------------

TOTAL ACROSS ALL RANGES:
  With Digesters:     154
  Without Digesters:  291
  Total:              445


In [7]:
# Cell 6: Perform Geographic Clustering
print("\n" + "="*80)
print("PERFORMING GEOGRAPHIC CLUSTERING WITHIN EACH HERD SIZE RANGE")
print("="*80)

all_clusters = []
cluster_counter = 0

for herd_range in herd_size_ranges:
    range_name = herd_range['name']
    range_data = facility_sjv[facility_sjv['HerdSizeRange'] == range_name].copy()
    
    if len(range_data) < 4:  # Skip ranges with too few dairies
        print(f"  • {range_name}: Skipped (only {len(range_data)} dairies)")
        continue
    
    # DBSCAN clustering within this herd size range
    coords = range_data[['Latitude', 'Longitude']].values
    dbscan = DBSCAN(eps=0.05, min_samples=3)  # ~5.5 km radius
    cluster_labels = dbscan.fit_predict(coords)
    
    range_data['ClusterID'] = cluster_labels
    
    # Filter out noise points (ClusterID == -1)
    range_data = range_data[range_data['ClusterID'] >= 0].copy()
    
    if len(range_data) == 0:
        print(f"  • {range_name}: No clusters found")
        continue
    
    # Reassign cluster IDs to be globally unique
    range_data['ClusterID'] = range_data['ClusterID'].apply(lambda x: x + cluster_counter)
    cluster_counter += int(range_data['ClusterID'].max()) + 1
    
    # Add to all_clusters
    all_clusters.append(range_data)
    
    num_clusters_in_range = int(range_data['ClusterID'].max() - range_data['ClusterID'].min() + 1)
    print(f"  ✓ {range_name}: {num_clusters_in_range} clusters with {len(range_data):,} dairies")

# Concatenate all cluster data
facility_sjv = pd.concat(all_clusters, ignore_index=True)
print(f"\n✓ Total: {len(facility_sjv):,} dairies across all clusters")


PERFORMING GEOGRAPHIC CLUSTERING WITHIN EACH HERD SIZE RANGE
  ✓ Small-Mid (1000-2000): 16 clusters with 172 dairies
  ✓ Mid (2000-3500): 12 clusters with 90 dairies
  ✓ Mid-Large (3000-5000): 2 clusters with 7 dairies
  • Large (4000-7000): No clusters found
  • Very Large (6000-10000): No clusters found

✓ Total: 269 dairies across all clusters


In [8]:
# Cell 7: Analyze Cluster Composition
print("\n" + "="*80)
print("ANALYZING CLUSTER COMPOSITION (DIGESTER/NON-DIGESTER PAIRS)")
print("="*80)

cluster_summary = []

for cluster_id in sorted(facility_sjv['ClusterID'].unique()):
    cluster_data = facility_sjv[facility_sjv['ClusterID'] == cluster_id].copy()
    
    digester_in_cluster = cluster_data[cluster_data['HasDigester']]
    non_digester_in_cluster = cluster_data[~cluster_data['HasDigester']]
    
    # Calculate centroid
    center_lat = cluster_data['Latitude'].mean()
    center_lon = cluster_data['Longitude'].mean()
    
    # Calculate average within-cluster distance (km)
    coords_cluster = cluster_data[['Latitude', 'Longitude']].values
    distances = cdist(coords_cluster, [[center_lat, center_lon]], metric='euclidean')
    avg_spacing_km = distances.mean() * 111  # 1 degree ≈ 111 km
    
    # Calculate herd size statistics
    avg_herd_size = cluster_data['MilkCows'].mean()
    herd_range = cluster_data['HerdSizeRange'].iloc[0]
    
    # Calculate herd size pairing quality
    digester_herd_mean = digester_in_cluster['MilkCows'].mean() if len(digester_in_cluster) > 0 else np.nan
    non_digester_herd_mean = non_digester_in_cluster['MilkCows'].mean() if len(non_digester_in_cluster) > 0 else np.nan
    herd_pairing_diff = abs(digester_herd_mean - non_digester_herd_mean) if (not np.isnan(digester_herd_mean) and not np.isnan(non_digester_herd_mean)) else np.nan
    
    cluster_summary.append({
        'ClusterID': int(cluster_id),
        'HerdSizeRange': herd_range,
        'NumDairies': len(cluster_data),
        'Digesters': len(digester_in_cluster),
        'NonDigesters': len(non_digester_in_cluster),
        'AvgHerdSize': int(avg_herd_size),
        'DigestorAvgHerd': int(digester_herd_mean) if not np.isnan(digester_herd_mean) else 0,
        'NonDigesterAvgHerd': int(non_digester_herd_mean) if not np.isnan(non_digester_herd_mean) else 0,
        'HerdPairingDiff': int(herd_pairing_diff) if not np.isnan(herd_pairing_diff) else 0,
        'AvgSpacingKm': round(avg_spacing_km, 1),
        'CenterLat': round(center_lat, 4),
        'CenterLon': round(center_lon, 4),
    })

cluster_summary_df = pd.DataFrame(cluster_summary)

print("\n" + "="*80)
print("CLUSTER SUMMARY TABLE")
print("="*80 + "\n")
print(cluster_summary_df.to_string(index=False))


ANALYZING CLUSTER COMPOSITION (DIGESTER/NON-DIGESTER PAIRS)

CLUSTER SUMMARY TABLE

 ClusterID         HerdSizeRange  NumDairies  Digesters  NonDigesters  AvgHerdSize  DigestorAvgHerd  NonDigesterAvgHerd  HerdPairingDiff  AvgSpacingKm  CenterLat  CenterLon
         0 Small-Mid (1000-2000)          36          9            27         1347             1284                1368               84           9.0    37.4363  -120.9326
         1 Small-Mid (1000-2000)           4          0             4         1396                0                1396                0           2.6    36.4094  -119.8584
         2 Small-Mid (1000-2000)           4          0             4         1624                0                1624                0           3.0    35.2664  -118.9471
         3 Small-Mid (1000-2000)           4          0             4         1174                0                1174                0           4.1    36.4283  -119.7549
         4 Small-Mid (1000-2000)          38      

In [9]:
# Cell 8: Identify Best Candidate Clusters
print("\n" + "="*80)
print("IDENTIFYING BEST CANDIDATE CLUSTERS")
print("="*80)

candidate_clusters = cluster_summary_df[
    (cluster_summary_df['Digesters'] >= 1) &
    (cluster_summary_df['NonDigesters'] >= 1) &
    (cluster_summary_df['AvgSpacingKm'] >= 1.5) &
    (cluster_summary_df['AvgSpacingKm'] <= 12)
].copy()

# Score on herd size pairing quality
candidate_clusters['PairingScore'] = candidate_clusters['HerdPairingDiff'].fillna(1000)
candidate_clusters = candidate_clusters.sort_values('PairingScore').reset_index(drop=True)

print(f"\n✓ Found {len(candidate_clusters)} candidate clusters meeting criteria:\n")

for idx, row in candidate_clusters.head(15).iterrows():
    pairing_quality = 'EXCELLENT' if row['HerdPairingDiff'] < 300 else 'GOOD' if row['HerdPairingDiff'] < 600 else 'OK'
    print(f"Candidate #{idx + 1} (Cluster ID: {int(row['ClusterID'])})")
    print(f"  Range: {row['HerdSizeRange']}")
    print(f"  Dairies: {int(row['NumDairies'])} total ({int(row['Digesters'])} digester, {int(row['NonDigesters'])} non-digester)")
    print(f"  Herd matching: Digesters avg {int(row['DigestorAvgHerd']):,} | Non-digesters avg {int(row['NonDigesterAvgHerd']):,}")
    print(f"  Pairing difference: {int(row['HerdPairingDiff']):,} cows ({pairing_quality})")
    print(f"  Spacing: {row['AvgSpacingKm']} km avg between dairies")
    print(f"  Center: ({row['CenterLat']}, {row['CenterLon']})")
    print()

print(f"\nTotal candidate clusters: {len(candidate_clusters)}")
print(f"Total dairies in candidates: {int(candidate_clusters['NumDairies'].sum())}")


IDENTIFYING BEST CANDIDATE CLUSTERS

✓ Found 16 candidate clusters meeting criteria:

Candidate #1 (Cluster ID: 9)
  Range: Small-Mid (1000-2000)
  Dairies: 4 total (1 digester, 3 non-digester)
  Herd matching: Digesters avg 1,750 | Non-digesters avg 1,701
  Pairing difference: 48 cows (EXCELLENT)
  Spacing: 2.3 km avg between dairies
  Center: (36.1996, -119.6959)

Candidate #2 (Cluster ID: 4)
  Range: Small-Mid (1000-2000)
  Dairies: 38 total (7 digester, 31 non-digester)
  Herd matching: Digesters avg 1,495 | Non-digesters avg 1,416
  Pairing difference: 78 cows (EXCELLENT)
  Spacing: 9.4 km avg between dairies
  Center: (36.0984, -119.294)

Candidate #3 (Cluster ID: 22)
  Range: Mid (2000-3500)
  Dairies: 8 total (4 digester, 4 non-digester)
  Herd matching: Digesters avg 2,408 | Non-digesters avg 2,327
  Pairing difference: 81 cows (EXCELLENT)
  Spacing: 3.1 km avg between dairies
  Center: (37.3948, -120.8792)

Candidate #4 (Cluster ID: 0)
  Range: Small-Mid (1000-2000)
  Dairie

In [10]:
# Cell 9: Export Candidate Dairies to CSV
print("\n" + "="*80)
print("EXPORTING CANDIDATE DAIRIES")
print("="*80)

candidate_cluster_ids = candidate_clusters['ClusterID'].values

export_cols = ['CADDID', 'FacilityName', 'City', 'County', 'Latitude', 'Longitude', 
               'StreetAddress', 'MilkCows', 'HerdSizeRange', 'HasDigester', 'ClusterID']

candidate_dairies = facility_sjv[facility_sjv['ClusterID'].isin(candidate_cluster_ids)][export_cols].copy()
candidate_dairies = candidate_dairies.sort_values(['ClusterID', 'HasDigester', 'MilkCows'], ascending=[True, False, False])

candidate_dairies.to_csv('SJV_Candidate_Dairies_for_Sampling.csv', index=False)

print(f"\n✓ Exported {len(candidate_dairies)} candidate dairies to:")
print(f"  'SJV_Candidate_Dairies_for_Sampling.csv'")
print(f"\nThis file is ready for Google Earth mapping!")
print(f"\nFile preview (first 10 rows):\n")
print(candidate_dairies.head(10).to_string(index=False))


EXPORTING CANDIDATE DAIRIES

✓ Exported 206 candidate dairies to:
  'SJV_Candidate_Dairies_for_Sampling.csv'

This file is ready for Google Earth mapping!

File preview (first 10 rows):

 CADDID         FacilityName          City     County  Latitude  Longitude             StreetAddress  MilkCows         HerdSizeRange  HasDigester  ClusterID
  11290      S & S Dairy Inc         Ceres Stanislaus   37.5222  -120.9895 348 East Monte Vista Road    1950.0 Small-Mid (1000-2000)         True          0
  10021  Albert Mendes Dairy Crows Landing Stanislaus   37.4852  -121.0086           1100 Ruble Road    1400.0 Small-Mid (1000-2000)         True          0
  10217          Dores Dairy     Stevinson     Merced   37.3354  -120.9025  22846 West Second Avenue    1305.0 Small-Mid (1000-2000)         True          0
  11312   K & R Blount Dairy Crows Landing Stanislaus   37.4854  -121.0033            624 Ruble Road    1240.0 Small-Mid (1000-2000)         True          0
  11129   Joe Oliveira Dair

In [11]:
# Cell 10: Detailed Dairy Information for Each Candidate Cluster
print("\n" + "="*80)
print("DETAILED DAIRY INFORMATION FOR CANDIDATE CLUSTERS")
print("="*80)

candidate_cluster_ids = candidate_clusters['ClusterID'].values

for cluster_idx, cluster_id in enumerate(candidate_cluster_ids[:5]):  # Show first 5 clusters
    cluster_data = facility_sjv[facility_sjv['ClusterID'] == cluster_id].sort_values('MilkCows', ascending=False)
    
    print(f"\n{'='*80}")
    print(f"CANDIDATE CLUSTER #{cluster_idx + 1} (ID: {int(cluster_id)})")
    print(f"Herd Size Range: {cluster_data['HerdSizeRange'].iloc[0]}")
    print(f"{'='*80}\n")
    
    # Sort by digester status first
    cluster_data_sorted = pd.concat([
        cluster_data[cluster_data['HasDigester']].sort_values('MilkCows', ascending=False),
        cluster_data[~cluster_data['HasDigester']].sort_values('MilkCows', ascending=False)
    ])
    
    for _, row in cluster_data_sorted.iterrows():
        digester_status = "✓ HAS DIGESTER" if row['HasDigester'] else "  NO DIGESTER"
        print(f"  {row['FacilityName']}")
        print(f"    {row['City']}, {row['County']} | ({row['Latitude']:.4f}, {row['Longitude']:.4f})")
        print(f"    Herd: {row['MilkCows']:,} milking cows | {digester_status}")
        print()

print("\nNote: Showing first 5 clusters. Run next cell for all clusters if needed.")


DETAILED DAIRY INFORMATION FOR CANDIDATE CLUSTERS

CANDIDATE CLUSTER #1 (ID: 9)
Herd Size Range: Small-Mid (1000-2000)

  ROCKING HORSE DAIRY LLC
    HANFORD, Kings | (36.1664, -119.6948)
    Herd: 1,750.0 milking cows | ✓ HAS DIGESTER

  Top Line Dairy #1
    Hanford, Kings | (36.2059, -119.6917)
    Herd: 1,930.0 milking cows |   NO DIGESTER

  Top Line Dairy #2
    Hanford, Kings | (36.2009, -119.6854)
    Herd: 1,700.0 milking cows |   NO DIGESTER

  Vaca Linda Dairy
    Hanford, Kings | (36.2252, -119.7116)
    Herd: 1,475.0 milking cows |   NO DIGESTER


CANDIDATE CLUSTER #2 (ID: 4)
Herd Size Range: Small-Mid (1000-2000)

  Little Rock Dairy
    Tipton, Tulare | (36.0401, -119.3958)
    Herd: 1,935.0 milking cows | ✓ HAS DIGESTER

  Schott Dairy
    Tipton, Tulare | (36.0393, -119.3532)
    Herd: 1,833.0 milking cows | ✓ HAS DIGESTER

  Rib-Arrow Dairy
    Tulare, Tulare | (36.1212, -119.2706)
    Herd: 1,500.0 milking cows | ✓ HAS DIGESTER

  Elk Creek Dairy
    Tulare, Tulare 

In [12]:
# Cell 11: Deployment Planning Summary
print("\n" + "="*80)
print("DEPLOYMENT PLANNING SUMMARY")
print("="*80)

print(f"\n✓ Total candidate clusters available: {len(candidate_clusters)}")
print(f"✓ Total candidate dairies: {int(candidate_clusters['NumDairies'].sum())}\n")

if len(candidate_clusters) >= 3:
    print("OPTION A: Use TOP 3-4 CLUSTERS (best herd size pairing)")
    print("-" * 80)
    top_clusters = candidate_clusters.head(min(4, len(candidate_clusters)))
    total_dairies_top = int(top_clusters['NumDairies'].sum())
    cluster_ids = ', '.join(map(str, map(int, top_clusters['ClusterID'].values)))
    print(f"  Clusters: {cluster_ids}")
    print(f"  Total dairies: {total_dairies_top}")
    print(f"  Digester/Non-digester: {int(top_clusters['Digesters'].sum())} / {int(top_clusters['NonDigesters'].sum())}")
    print(f"  Herd size ranges: {', '.join(top_clusters['HerdSizeRange'].unique())}\n")

if len(candidate_clusters) >= 5:
    print("OPTION B: Use TOP 5 CLUSTERS (maximum diversity)")
    print("-" * 80)
    best_5 = candidate_clusters.head(5)
    total_dairies_best5 = int(best_5['NumDairies'].sum())
    cluster_ids = ', '.join(map(str, map(int, best_5['ClusterID'].values)))
    print(f"  Clusters: {cluster_ids}")
    print(f"  Total dairies: {total_dairies_best5}")
    print(f"  Digester/Non-digester: {int(best_5['Digesters'].sum())} / {int(best_5['NonDigesters'].sum())}")
    print(f"  Herd size ranges: {', '.join(best_5['HerdSizeRange'].unique())}\n")

print("NEXT STEPS:")
print("-" * 80)
print("1. ✓ Review 'SJV_Candidate_Dairies_for_Sampling.csv'")
print("2. ✓ Import coordinates into Google Earth")
print("3. ✓ Assess road access for SE transects (NW prevailing winds)")
print("4. ✓ Verify upwind background monitor feasibility")
print("5. ✓ Contact facility operators for site access")
print("6. ✓ Finalize 15-20 dairy roster\n")

print("="*80)
print("ANALYSIS COMPLETE!")
print("="*80)


DEPLOYMENT PLANNING SUMMARY

✓ Total candidate clusters available: 16
✓ Total candidate dairies: 206

OPTION A: Use TOP 3-4 CLUSTERS (best herd size pairing)
--------------------------------------------------------------------------------
  Clusters: 9, 4, 22, 0
  Total dairies: 86
  Digester/Non-digester: 21 / 65
  Herd size ranges: Small-Mid (1000-2000), Mid (2000-3500)

OPTION B: Use TOP 5 CLUSTERS (maximum diversity)
--------------------------------------------------------------------------------
  Clusters: 9, 4, 22, 0, 5
  Total dairies: 101
  Digester/Non-digester: 24 / 77
  Herd size ranges: Small-Mid (1000-2000), Mid (2000-3500)

NEXT STEPS:
--------------------------------------------------------------------------------
1. ✓ Review 'SJV_Candidate_Dairies_for_Sampling.csv'
2. ✓ Import coordinates into Google Earth
3. ✓ Assess road access for SE transects (NW prevailing winds)
4. ✓ Verify upwind background monitor feasibility
5. ✓ Contact facility operators for site access
6. 

In [18]:
# Cell 14 (REVISED): Strategic Site Filtering - Distance-Weighted Upwind Interference & Herd Size Matching
import math

print("\n" + "="*80)
print("STRATEGIC SITE FILTERING: WIND GEOMETRY, UPWIND INTERFERENCE & HERD SIZE MATCHING")
print("="*80)

# Load the candidate dairies CSV
candidate_dairies_full = pd.read_csv('SJV_Candidate_Dairies_for_Sampling.csv')

print(f"\nLoaded {len(candidate_dairies_full)} candidate dairies")
print(f"Clusters represented: {candidate_dairies_full['ClusterID'].nunique()}")

# ==================== DISTANCE & BEARING CALCULATIONS ====================

def calculate_bearing(lat1, lon1, lat2, lon2):
    """
    Calculate bearing from point 1 to point 2
    Returns bearing in degrees (0-360)
    0° = North, 90° = East, 180° = South, 270° = West
    """
    dLon = lon2 - lon1
    y = math.sin(math.radians(dLon)) * math.cos(math.radians(lat2))
    x = (math.cos(math.radians(lat1)) * math.sin(math.radians(lat2)) - 
         math.sin(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.cos(math.radians(dLon)))
    bearing = math.degrees(math.atan2(y, x))
    return (bearing + 360) % 360

def calculate_distance_km(lat1, lon1, lat2, lon2):
    """Calculate distance between two points in km using Haversine formula"""
    R = 6371  # Earth's radius in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

# ==================== UPWIND INTERFERENCE SCORING ====================

def calculate_upwind_interference_score(target_dairy, cluster_dairies):
    """
    Calculate distance-weighted upwind interference score
    
    Scoring:
    - Upwind dairy < 1 km away = 20 points penalty
    - Upwind dairy 1-2 km away = 12 points penalty
    - Upwind dairy 2-3 km away = 8 points penalty
    - Upwind dairy 3-5 km away = 4 points penalty
    - Upwind dairy > 5 km away = 0 points penalty
    
    Returns: upwind_score (lower is better, 0 is best)
    """
    
    target_lat = target_dairy['Latitude']
    target_lon = target_dairy['Longitude']
    
    other_dairies = cluster_dairies[
        (cluster_dairies['Latitude'] != target_lat) | (cluster_dairies['Longitude'] != target_lon)
    ]
    
    upwind_score = 0
    upwind_details = []
    
    for _, other in other_dairies.iterrows():
        bearing = calculate_bearing(target_lat, target_lon, other['Latitude'], other['Longitude'])
        distance = calculate_distance_km(target_lat, target_lon, other['Latitude'], other['Longitude'])
        
        # Check if in NW quadrant (upwind for NW prevailing winds: bearing 270-45)
        upwind_bearing = (bearing >= 270 or bearing <= 45)
        
        if upwind_bearing:
            # Distance-weighted penalty
            if distance < 1.0:
                penalty = 20
            elif distance < 2.0:
                penalty = 12
            elif distance < 3.0:
                penalty = 8
            elif distance < 5.0:
                penalty = 4
            else:
                penalty = 0
            
            upwind_score += penalty
            upwind_details.append({
                'facility': other['FacilityName'],
                'distance_km': distance,
                'bearing': bearing,
                'penalty': penalty
            })
    
    return upwind_score, upwind_details

# ==================== HERD SIZE MATCHING SCORING ====================

def calculate_herd_size_matching_score(target_dairy, cluster_dairies):
    """
    Calculate herd size matching score for digester/non-digester comparison
    
    PRIMARY: Milk cows similarity (most important)
    SECONDARY: Total herd size similarity (bonus)
    
    Scoring:
    - Milk cow difference < 100 = 0 points (excellent match)
    - Milk cow difference 100-300 = 2 points
    - Milk cow difference 300-600 = 5 points
    - Milk cow difference > 600 = 10 points
    
    BONUS for similar total herd size:
    - Total herd difference < 200 = -2 bonus points (good match)
    """
    
    target_milk_cows = target_dairy['MilkCows']
    target_digester = target_dairy['HasDigester']
    
    # Find matching dairy (opposite digester status)
    opposite_dairies = cluster_dairies[
        cluster_dairies['HasDigester'] != target_digester
    ]
    
    if len(opposite_dairies) == 0:
        return 100, None  # No match available
    
    best_match_score = float('inf')
    best_match_dairy = None
    
    for _, other in opposite_dairies.iterrows():
        other_milk_cows = other['MilkCows']
        
        # PRIMARY: Milk cow matching
        milk_cow_diff = abs(target_milk_cows - other_milk_cows)
        
        if milk_cow_diff < 100:
            milk_cow_score = 0
        elif milk_cow_diff < 300:
            milk_cow_score = 2
        elif milk_cow_diff < 600:
            milk_cow_score = 5
        else:
            milk_cow_score = 10
        
        # SECONDARY: Total herd size bonus
        # Calculate total herd for both
        target_total = (target_dairy.get('MilkCows', 0) + 
                       target_dairy.get('DryCows', 0) + 
                       target_dairy.get('OldHeifers', 0) + 
                       target_dairy.get('YoungHeifers', 0))
        
        other_total = (other.get('MilkCows', 0) + 
                      other.get('DryCows', 0) + 
                      other.get('OldHeifers', 0) + 
                      other.get('YoungHeifers', 0))
        
        total_herd_diff = abs(target_total - other_total)
        total_herd_bonus = 0 if total_herd_diff < 200 else -2
        
        # Combined score (lower is better)
        combined_score = milk_cow_score + total_herd_bonus
        
        if combined_score < best_match_score:
            best_match_score = combined_score
            best_match_dairy = {
                'facility': other['FacilityName'],
                'milk_cows': other_milk_cows,
                'milk_cow_diff': milk_cow_diff,
                'milk_cow_score': milk_cow_score,
                'total_herd_diff': total_herd_diff,
                'total_herd_bonus': total_herd_bonus,
                'combined_score': combined_score
            }
    
    return best_match_score, best_match_dairy

# ==================== COMBINED SAMPLING QUALITY SCORE ====================

def assess_sampling_quality_v2(target_dairy, cluster_dairies):
    """
    Calculate overall sampling quality score
    
    Components:
    1. Upwind interference (distance-weighted)
    2. Herd size matching quality
    3. Digester status bonus
    
    Final score: 0-100 (higher is better)
    """
    
    # Get upwind interference score
    upwind_score, upwind_details = calculate_upwind_interference_score(target_dairy, cluster_dairies)
    
    # Get herd size matching score
    herd_matching_score, matching_dairy = calculate_herd_size_matching_score(target_dairy, cluster_dairies)
    
    # Base score starts at 100
    quality_score = 100
    
    # Penalize upwind interference
    quality_score -= upwind_score
    
    # Penalize poor herd matching (but cap the penalty)
    quality_score -= min(herd_matching_score * 2, 30)
    
    # Bonus for digesters (more interesting for research)
    if target_dairy['HasDigester']:
        quality_score += 10
    
    # Ensure score stays in 0-100 range
    quality_score = max(0, min(100, quality_score))
    
    return {
        'quality_score': round(quality_score, 1),
        'upwind_interference_score': upwind_score,
        'herd_matching_score': herd_matching_score,
        'matching_dairy': matching_dairy,
        'upwind_details': upwind_details
    }

# ==================== APPLY ASSESSMENT TO ALL DAIRIES ====================

print("\nAssessing sampling quality for each dairy...")
print("This includes:")
print("  • Distance-weighted upwind interference")
print("  • Milk cow herd size matching")
print("  • Total herd size similarity (bonus)")
print("\n")

assessment_results = []

for _, dairy in candidate_dairies_full.iterrows():
    cluster_dairies = candidate_dairies_full[candidate_dairies_full['ClusterID'] == dairy['ClusterID']]
    assessment = assess_sampling_quality_v2(dairy, cluster_dairies)
    
    assessment_results.append({
        'CADDID': dairy['CADDID'],
        'QualityScore': assessment['quality_score'],
        'UpwindInterferenceScore': assessment['upwind_interference_score'],
        'HerdMatchingScore': assessment['herd_matching_score'],
        'MatchingFacility': assessment['matching_dairy']['facility'] if assessment['matching_dairy'] else 'None',
        'MatchingMilkCowDiff': assessment['matching_dairy']['milk_cow_diff'] if assessment['matching_dairy'] else np.nan,
        'MatchingTotalHerdDiff': assessment['matching_dairy']['total_herd_diff'] if assessment['matching_dairy'] else np.nan
    })

assessment_df = pd.DataFrame(assessment_results)
candidate_dairies_full = candidate_dairies_full.merge(assessment_df, on='CADDID', how='left')

print("✓ Assessment complete!\n")

# ==================== DISPLAY SAMPLE RESULTS ====================

print("SAMPLE ASSESSMENT RESULTS (First 20 dairies):")
print("="*80)

display_cols = ['FacilityName', 'MilkCows', 'HasDigester', 'ClusterID', 
                'QualityScore', 'UpwindInterferenceScore', 'MatchingFacility', 'MatchingMilkCowDiff']

sample_display = candidate_dairies_full[display_cols].head(20).copy()
sample_display['QualityScore'] = sample_display['QualityScore'].round(1)
sample_display['MatchingMilkCowDiff'] = sample_display['MatchingMilkCowDiff'].round(0).astype('Int64')

print(sample_display.to_string(index=False))

print("\n" + "="*80)
print("SCORING EXPLANATION:")
print("="*80)
print("""
QUALITY SCORE (0-100, higher is better):
  • Based on: Upwind interference + Herd size matching + Digester bonus
  
UPWIND INTERFERENCE SCORE (lower is better):
  • < 1 km upwind:   20 points penalty
  • 1-2 km upwind:   12 points penalty
  • 2-3 km upwind:    8 points penalty
  • 3-5 km upwind:    4 points penalty
  • > 5 km upwind:    0 points penalty
  
HERD MATCHING SCORE (milk cows - PRIMARY focus):
  • Difference < 100 cows:     0 points (excellent)
  • Difference 100-300 cows:   2 points (good)
  • Difference 300-600 cows:   5 points (fair)
  • Difference > 600 cows:    10 points (poor)
  
BONUS (Secondary focus - total herd):
  • Total herd difference < 200: -2 bonus points
""")



STRATEGIC SITE FILTERING: WIND GEOMETRY, UPWIND INTERFERENCE & HERD SIZE MATCHING

Loaded 206 candidate dairies
Clusters represented: 16

Assessing sampling quality for each dairy...
This includes:
  • Distance-weighted upwind interference
  • Milk cow herd size matching
  • Total herd size similarity (bonus)


✓ Assessment complete!

SAMPLE ASSESSMENT RESULTS (First 20 dairies):
         FacilityName  MilkCows  HasDigester  ClusterID  QualityScore  UpwindInterferenceScore        MatchingFacility  MatchingMilkCowDiff
      S & S Dairy Inc    1950.0         True          0           100                        4         Moonshine Dairy                   50
  Albert Mendes Dairy    1400.0         True          0            90                       20               B-6 Dairy                  252
          Dores Dairy    1305.0         True          0           100                        4           Dairy Central                  295
   K & R Blount Dairy    1240.0         True          0 

In [19]:
# Cell 15: Select Best 4 Dairies Per Cluster
print("\n" + "="*80)
print("SELECTING TOP 4 DAIRIES PER CLUSTER")
print("="*80)

# For each cluster, select the best 4 dairies
# Criteria: 
# 1. At least 1-2 digesters, 2-3 non-digesters
# 2. High sampling score (low upwind interference)
# 3. Good herd size representation

top_4_per_cluster = []

for cluster_id in sorted(candidate_dairies_full['ClusterID'].unique()):
    cluster_data = candidate_dairies_full[candidate_dairies_full['ClusterID'] == cluster_id].copy()
    
    # Sort by: digester status (digesters first), then sampling score
    cluster_data = cluster_data.sort_values(
        ['HasDigester', 'SamplingScore'], 
        ascending=[False, False]
    )
    
    # Try to get 2 digesters + 2 non-digesters if possible
    digesters = cluster_data[cluster_data['HasDigester']].head(2)
    non_digesters = cluster_data[~cluster_data['HasDigester']].head(2)
    
    selected = pd.concat([digesters, non_digesters])
    selected = selected.sort_values('SamplingScore', ascending=False)
    
    # If we don't have enough, fill with next best options
    if len(selected) < 4:
        all_others = cluster_data[~cluster_data['CADDID'].isin(selected['CADDID'])]
        needed = 4 - len(selected)
        selected = pd.concat([selected, all_others.head(needed)])
    
    selected = selected.head(4)
    selected['SelectionRank'] = range(1, len(selected) + 1)
    selected['ClusterRank'] = cluster_id
    
    top_4_per_cluster.append(selected)

# Combine all selected dairies
top_4_df = pd.concat(top_4_per_cluster, ignore_index=True)
top_4_df = top_4_df.sort_values(['ClusterID', 'SelectionRank'])

print(f"\n✓ Selected {len(top_4_df)} dairies ({len(top_4_df)//4} clusters × 4 dairies)")

# Display summary by cluster
print("\nSUMMARY OF SELECTED DAIRIES BY CLUSTER:")
print("="*80)

for cluster_id in sorted(top_4_df['ClusterID'].unique()):
    cluster_selection = top_4_df[top_4_df['ClusterID'] == cluster_id]
    herd_range = cluster_selection['HerdSizeRange'].iloc[0]
    num_digesters = cluster_selection['HasDigester'].sum()
    
    print(f"\n🔹 CLUSTER {int(cluster_id)} ({herd_range})")
    print(f"   Digesters: {int(num_digesters)}/4 | Sampling quality: GOOD")
    
    for idx, (_, row) in enumerate(cluster_selection.iterrows(), 1):
        digester_label = "✓" if row['HasDigester'] else " "
        print(f"   {idx}. {digester_label} {row['FacilityName'][:45]:45} | {int(row['MilkCows']):,} head | Score: {row['SamplingScore']}")

# Export to CSV
export_cols = ['CADDID', 'FacilityName', 'City', 'County', 'Latitude', 'Longitude', 
               'StreetAddress', 'MilkCows', 'HerdSizeRange', 'HasDigester', 'ClusterID',
               'UpwindDairies', 'NearbyDairies', 'SamplingScore', 'SelectionRank']

top_4_export = top_4_df[export_cols].copy()
top_4_export.to_csv('SJV_Top4_Best_Sites_by_Cluster.csv', index=False)

print(f"\n\n{'='*80}")
print("✓ Exported to: 'SJV_Top4_Best_Sites_by_Cluster.csv'")
print("="*80)

print(f"\n📊 FINAL PORTFOLIO SUMMARY:")
print(f"   Total dairies selected: {len(top_4_df)}")
print(f"   Clusters: {top_4_df['ClusterID'].nunique()}")
print(f"   Digesters: {int(top_4_df['HasDigester'].sum())}")
print(f"   Non-digesters: {len(top_4_df) - int(top_4_df['HasDigester'].sum())}")
print(f"   Herd size ranges: {', '.join(top_4_df['HerdSizeRange'].unique())}")


SELECTING TOP 4 DAIRIES PER CLUSTER


KeyError: 'SamplingScore'

In [20]:
# Cell 16: Export to Google Earth KML Format with Cluster Labels
import xml.sax.saxutils as saxutils

print("\n" + "="*80)
print("CREATING GOOGLE EARTH KML FILE WITH CLUSTER INFORMATION")
print("="*80)

kml_header = """<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
  <Document>
    <name>SJV Dairy Sampling Sites - Clustered</name>
    <description>Best sampling sites for mobile emissions measurement - Organized by cluster</description>
    
    <Style id="digester">
      <IconStyle>
        <color>ff0000ff</color>
        <scale>1.2</scale>
      </IconStyle>
      <LabelStyle>
        <scale>1.0</scale>
      </LabelStyle>
    </Style>
    
    <Style id="non-digester">
      <IconStyle>
        <color>ff00ff00</color>
        <scale>1.0</scale>
      </IconStyle>
      <LabelStyle>
        <scale>1.0</scale>
      </LabelStyle>
    </Style>
"""

# Group dairies by cluster
clusters = top_4_df['ClusterID'].unique()
clusters = sorted(clusters)

kml_folders = ""

for cluster_id in clusters:
    cluster_data = top_4_df[top_4_df['ClusterID'] == cluster_id]
    herd_range = cluster_data['HerdSizeRange'].iloc[0]
    num_digesters = int(cluster_data['HasDigester'].sum())
    num_non_digesters = len(cluster_data) - num_digesters
    
    # Create folder for each cluster
    folder_start = f"""    <Folder>
      <name>Cluster {int(cluster_id)} - {herd_range}</name>
      <description>
        {int(len(cluster_data))} dairies ({num_digesters} digesters, {num_non_digesters} non-digesters)
      </description>
      <open>1</open>
"""
    
    kml_placemarks = ""
    
    for idx, (_, row) in enumerate(cluster_data.iterrows(), 1):
        style_id = "digester" if row['HasDigester'] else "non-digester"
        icon_name = "[DIGESTER]" if row['HasDigester'] else "[Non-Digester]"
        
        # Escape XML special characters
        facility_name = saxutils.escape(str(row['FacilityName']))
        city = saxutils.escape(str(row['City']))
        county = saxutils.escape(str(row['County']))
        address = saxutils.escape(str(row['StreetAddress']))
        matching_facility = saxutils.escape(str(row['MatchingFacility'])) if pd.notna(row['MatchingFacility']) else "N/A"
        
        # Create label with cluster ID and dairy name
        label = f"C{int(cluster_id)}-{idx}: {facility_name}"
        
        placemark = f"""      <Placemark>
        <name>{label}</name>
        <description>
          {icon_name}
          
          Facility: {facility_name}
          Cluster: {int(cluster_id)}
          Rank in Cluster: {int(idx)}/4
          
          Herd Information:
          - Milking Cows: {int(row['MilkCows']):,}
          - Location: {city}, {county}
          - Address: {address}
          
          Sampling Quality Scores:
          - Overall Quality Score: {row['QualityScore']}
          - Upwind Interference Score: {int(row['UpwindInterferenceScore'])}
          - Herd Matching Score: {int(row['HerdMatchingScore'])}
          
          Digester Comparison:
          - Best Match: {matching_facility}
          - Milk Cow Difference: {int(row['MatchingMilkCowDiff']) if pd.notna(row['MatchingMilkCowDiff']) else 'N/A'} cows
          - Total Herd Difference: {int(row['MatchingTotalHerdDiff']) if pd.notna(row['MatchingTotalHerdDiff']) else 'N/A'}
        </description>
        <styleUrl>#{style_id}</styleUrl>
        <Point>
          <coordinates>{row['Longitude']},{row['Latitude']},0</coordinates>
        </Point>
      </Placemark>
"""
        kml_placemarks += placemark
    
    folder_end = """    </Folder>
"""
    
    kml_folders += folder_start + kml_placemarks + folder_end

kml_footer = """  </Document>
</kml>
"""

kml_content = kml_header + kml_folders + kml_footer

# Write with UTF-8 encoding
try:
    with open('SJV_Dairy_Sampling_Sites.kml', 'w', encoding='utf-8') as f:
        f.write(kml_content)
    print("\n✓ Successfully created: 'SJV_Dairy_Sampling_Sites.kml'")
except Exception as e:
    print(f"\n✗ Error creating KML: {e}")

print("\n" + "="*80)
print("KML FILE STRUCTURE:")
print("="*80)
print(f"\nTotal clusters: {len(clusters)}")
print(f"Total dairies: {len(top_4_df)}")

for cluster_id in clusters:
    cluster_data = top_4_df[top_4_df['ClusterID'] == cluster_id]
    herd_range = cluster_data['HerdSizeRange'].iloc[0]
    num_digesters = int(cluster_data['HasDigester'].sum())
    print(f"  • Cluster {int(cluster_id)} ({herd_range}): {num_digesters} digesters, {len(cluster_data)-num_digesters} non-digesters")

print("\n" + "="*80)
print("HOW TO USE IN GOOGLE EARTH:")
print("="*80)
print("""
1. Open Google Earth Pro
2. File → Import → Select 'SJV_Dairy_Sampling_Sites.kml'
3. In left panel, expand each CLUSTER folder to see dairies
4. Pin Labels show: C[ClusterID]-[RankInCluster]: [DairyName]
   Example: "C9-1: Rocking Horse Dairy LLC"
   Meaning: Cluster 9, Rank 1 (best in cluster)

5. Click each pin to see detailed information:
   - Quality scores for each site
   - Best digester/non-digester match
   - Upwind interference assessment
   - Herd size differences

6. Use Google Earth to visually verify:
   - Road access (can you reach SE transect points?)
   - Topography (any wind obstruction?)
   - Background monitor location (upwind area is clear?)
   - Cluster spatial distribution

PIN COLORS:
  BLUE  = Digester dairies (higher emissions expected)
  GREEN = Non-digester dairies (lower emissions expected)
""")

print("\n" + "="*80)
print("DEPLOYMENT PLANNING:")
print("="*80)
print(f"""
You now have {len(top_4_df)} dairies across {len(clusters)} clusters.
Each cluster contains 4 carefully selected sites based on:
  ✓ Minimal upwind interference (distance-weighted)
  ✓ Similar milk cow herd sizes (primary comparison metric)
  ✓ Similar total herd sizes (secondary bonus)
  ✓ Good digester/non-digester pairing

For your 3 deployments:
  - Spring: Visit ALL clusters (all {len(top_4_df)} dairies)
  - Summer: Visit same clusters (same {len(top_4_df)} dairies)
  - Fall: Visit same clusters (same {len(top_4_df)} dairies)
  
This provides seasonal variability while controlling for site-specific factors.
""")

print("\n✓ Ready for Google Earth visualization!\n")


CREATING GOOGLE EARTH KML FILE WITH CLUSTER INFORMATION


KeyError: 'MatchingFacility'